In [1]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

/Users/chenjinghe/Desktop/python-projects/mkpdf/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PDF to HTML

In [2]:
source = "../paper/JME_DL.pdf"

# 最快設定：JME PDF 是 born-digital 有 text layer，OCR 不需要；
# 公式 / 圖片 enrichment 都關掉，後面 VLM 階段再處理。
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.do_formula_enrichment = False
pipeline_options.generate_picture_images = True  # 圖片還是要進 HTML

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=StandardPdfPipeline,
            pipeline_options=pipeline_options,
        ),
    }
)

result = converter.convert(source)
print(result.document.export_to_markdown()[:2000])

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 4720.74it/s]


<!-- image -->

Contents lists available at ScienceDirect

## Journal of Monetary Economics

[journal homepage: www.elsevier.com/locate/jmoneco](http://www.elsevier.com/locate/jmoneco)

## Deep learning for solving dynamic economic models.

Lilia Maliar a , Serguei Maliar b , ∗ , Pablo Winant c

- a The Graduate Center, City University of New York, CEPR, and Hoover Institution, Stanford University

b Santa Clara University

c ESCP Business School and CREST/Ecole Polytechnique

## a r t i c l e i n f o

Article history: Received 11 March 2020 Revised 16 July 2021 Accepted 19 July 2021 Available online 21 July 2021

JEL classification:

C61

C63

C65

C68

C88

E32

E37

Keywords: Artificial intelligence Machine learning Deep learning Neural network Stochastic gradient Dynamic models Model reduction Dynamic programming Bellman equation Euler equation

Value functio

## 1. Introduction

Artificial intelligence (AI) has remarkable applications, such as recognition of images and speech, fac

In [4]:
from pathlib import Path

from docling_core.types.doc import ImageRefMode

# 改寫到新資料夾，VLM 的結果 (../outputs-test) 保留不覆蓋方便對比
output_dir = Path("../outputs-test-std")
output_dir.mkdir(exist_ok=True)

referenced_html = output_dir / "JME_DL_referenced.html"
embedded_html = output_dir / "JME_DL_embedded.html"
split_page_html = output_dir / "JME_DL_split_page.html"

result.document.save_as_html(
    referenced_html,
    image_mode=ImageRefMode.REFERENCED,
)

result.document.save_as_html(
    embedded_html,
    image_mode=ImageRefMode.EMBEDDED,
)

result.document.save_as_html(
    split_page_html,
    image_mode=ImageRefMode.REFERENCED,
    split_page_view=True,
)

print(f"Saved: {referenced_html}")
print(f"Saved: {embedded_html}")
print(f"Saved: {split_page_html}")

# Sanity check: 比對原 PDF 頁數 vs HTML 產生的 page divs，以及有沒有空頁
import re
import pypdfium2 as pdfium

pdf_pages = len(pdfium.PdfDocument(source))
html = split_page_html.read_text()
indices = [m.start() for m in re.finditer(r"<div class='page'>", html)]
indices.append(html.find("</body>"))
empties = []
for i in range(len(indices) - 1):
    chunk = html[indices[i]:indices[i + 1]]
    text = re.sub(r"<[^>]+>", " ", chunk)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) == 0:
        empties.append(i + 1)

print(f"\nPDF pages:       {pdf_pages}")
print(f"HTML page divs:  {len(indices) - 1}")
print(f"Empty page divs: {empties or 'none'}")

Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with

Saved: ../outputs-test-std/JME_DL_referenced.html
Saved: ../outputs-test-std/JME_DL_embedded.html
Saved: ../outputs-test-std/JME_DL_split_page.html

PDF pages:       26
HTML page divs:  26
Empty page divs: none


# 2. Then